In [ ]:
# thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import ParameterGrid
import pmdarima as pm
from pmdarima import model_selection

In [ ]:
# tải dữ liệu
# YYYY-MM-DD (ngày đầu tháng)
df = pd.read_csv('../tnbike_data.csv', index_col=0, parse_dates=True)
df.head(0)

In [ ]:
# đổi tên biến
df = df.rename(columns={'revenue': 'y'})
df.head(1)

In [ ]:
# trích xuất biến phụ (regressors)
# tháng, nhóm sản phẩm, ngày lễ Tết
X = df.iloc[:, 1:]
X.head(0)

# Tính dừng

In [ ]:
# kiểm định ADF
from statsmodels.tsa.stattools import adfuller
pvalue = adfuller(x=df.y)[1]

# đọc kết quả
if pvalue < 0.05:
    print(f"Chuỗi thời gian DỪNG. P-Value = {pvalue:.3f}")
else:
    print(f"Chuỗi thời gian KHÔNG DỪNG. P-Value = {pvalue:.3f}")

In [ ]:
# lấy sai phân bậc 1
pvalue = adfuller(df.y.diff().dropna())[1]

# đọc kết quả
if pvalue < 0.05:
    print(f"Chuỗi thời gian DỪNG sau sai phân. P-Value = {pvalue:.3f}")
else:
    print(f"Chuỗi thời gian KHÔNG DỪNG sau sai phân. P-Value = {pvalue:.3f}")

# Mô hình SARIMAX

In [ ]:
# mô hình
# tần suất tháng → seasonal = 12
model = pm.ARIMA(order=(1, 1, 1),
                 seasonal_order=(1, 1, 1, 12),
                 suppress_warnings=True,
                 force_stationarity=False)

In [ ]:
# kiểm định chéo (cross-validation)
cv = model_selection.RollingForecastCV(h=3,
                                       step=2,
                                       initial=df.shape[0] - 12)
cv_score = model_selection.cross_val_score(model,
                                           y=df.y,
                                           X=X,
                                           scoring='mean_squared_error',
                                           cv=cv,
                                           verbose=2,
                                           error_score=10000000000000)

In [ ]:
# hiệu suất CV
np.sqrt(np.average(cv_score))

# Tinh chỉnh tham số

In [ ]:
# lưới tham số
param_grid = {'p': [0, 1],
              'd': [0, 1],
              'q': [0, 1],
              'P': [0, 1],
              'D': [0, 1],
              'Q': [0, 1]}
grid = ParameterGrid(param_grid)
len(list(grid))

In [ ]:
# vòng lặp tinh chỉnh tham số
rmse = []
i = 1
# duyệt từng bộ tham số
for params in grid:
    print(f"{i} / {len(list(grid))}")
    # mô hình
    model = pm.ARIMA(order=(params['p'], params['d'], params['q']),
                     seasonal_order=(params['P'], params['D'], params['Q'], 12),
                     suppress_warnings=True,
                     force_stationarity=False)
    # CV
    cv = model_selection.RollingForecastCV(h=3,
                                           step=2,
                                           initial=df.shape[0] - 12)
    cv_score = model_selection.cross_val_score(model,
                                               y=df.y,
                                               X=X,
                                               scoring='mean_squared_error',
                                               cv=cv,
                                               verbose=2,
                                               error_score=100000)
    # sai số
    error = np.sqrt(np.average(cv_score))
    rmse.append(error)
    i += 1

In [ ]:
tuning_results = pd.DataFrame(grid)
tuning_results['rmse'] = rmse
tuning_results

In [ ]:
# xuất tham số tốt nhất
best_params = tuning_results[tuning_results.rmse == tuning_results.rmse.min()].transpose()
best_params.to_csv("../Forecasting Product/best_params_sarimax.csv")
best_params